# Test MCP Server
Set your MCP server route URL below, then run the cells.

In [ ]:
%pip install mcp==1.26.0

In [ ]:
MCP_ENDPOINT = "https://<YOUR-MCP-ROUTE>/mcp"

## Discover available tools

In [ ]:
from mcp.client.streamable_http import streamablehttp_client
from mcp import ClientSession

async with streamablehttp_client(MCP_ENDPOINT) as (read, write, _):
    async with ClientSession(read, write) as session:
        await session.initialize()
        tools = await session.list_tools()
        for tool in tools.tools:
            print(f"Tool: {tool.name}")
            print(f"  Description: {tool.description}")
            print(f"  Parameters:")
            for name, prop in tool.inputSchema.get("properties", {}).items():
                print(f"    {name}: {prop.get('type', '?')}")

In [ ]:
async def call_tool(sample: dict) -> str:
    async with streamablehttp_client(MCP_ENDPOINT) as (read, write, _):
        async with ClientSession(read, write) as session:
            await session.initialize()
            result = await session.call_tool("check_loan_approval", sample)
            return result.content[0].text

## Positive sample (expect: approved)

In [ ]:
pos_sample = {
    "no_of_dependents":          2,
    "graduated":                 True,
    "self_employed":             False,
    "income_annum":              9600000,
    "loan_amount":               29900000,
    "loan_term":                 12,
    "cibil_score":               778,
    "residential_assets_value":  2400000,
    "commercial_assets_value":   17600000,
    "luxury_assets_value":       22700000,
    "bank_asset_value":          8000000,
}

print(await call_tool(pos_sample))

## Negative sample (expect: rejected)

In [ ]:
neg_sample = {
    "no_of_dependents":          0,
    "graduated":                 False,
    "self_employed":             False,
    "income_annum":              2000000,
    "loan_amount":               4800000,
    "loan_term":                 20,
    "cibil_score":               337,
    "residential_assets_value":  1300000,
    "commercial_assets_value":   2200000,
    "luxury_assets_value":       5500000,
    "bank_asset_value":          1700000,
}

print(await call_tool(neg_sample))